# 05. TTS 테스트 (Qwen3-TTS) — **Colab GPU**

확인할 것
1. 실존 인물이 아닌 **가상의 캐스터 목소리**를 VoiceDesign으로 만들 수 있는가
2. 그 목소리를 Base(음성 복제)로 고정해서 대본 전체를 **같은 목소리**로 읽을 수 있는가
3. 숫자(151킬로, 2루타), 야구 용어(투앤원, 풀카운트, 스위퍼), 선수 이름을 제대로 읽는가
4. **속도(RTF)** → 클립 몇 개까지 합성할 수 있는가

모델: Qwen3-TTS (Apache 2.0, 한국어 지원)

In [ ]:
!pip install -q -U qwen-tts
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import gc
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

OUT = Path("tts_test")
OUT.mkdir(exist_ok=True)


def load(name):
    # flash-attn 빌드는 오래 걸려서 PyTorch 기본 어텐션(sdpa)으로 시작
    return Qwen3TTSModel.from_pretrained(name, device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa")


def free():
    """호출하는 쪽에서 `del 모델` 한 뒤에 부른다 (함수 안에서 del 해봐야 바깥 변수는 남아 있음)"""
    gc.collect()
    torch.cuda.empty_cache()


class Timer:
    def __enter__(self):
        torch.cuda.synchronize()
        self.t = time.perf_counter()
        return self

    def __exit__(self, *_):
        torch.cuda.synchronize()
        self.sec = time.perf_counter() - self.t

In [ ]:
# 1) 가상의 캐스터 목소리 설계. 참조 문장은 직접 쓴 일반적인 중계 오프닝 (실제 중계 원문 아님)
# VoiceDesign은 실행할 때마다 다른 목소리를 만든다
#  → 이미 저장된 참조 음성은 다시 만들지 않고, 새로 만드는 목소리는 시드를 고정한다
#  → 설명을 바꿔서 다시 뽑을 때는 이름(버전)을 올린다 (같은 이름이면 예전 파일을 불러옴)
REF_TEXT = "안녕하십니까, 오늘도 프로야구 중계로 함께하겠습니다. 양 팀 선발 투수의 맞대결, 시작부터 기대가 되는데요."

VOICES = {  # 이름: (설명, 시드). 남성 캐스터만
    "caster_m30": ("A Korean male sports play-by-play announcer in his 30s, clear and energetic voice, "
                   "fast and crisp delivery, broadcasting a live baseball game.", 0),  # 확정
    # v1 톤은 좋았으나 문장 끝을 인위적으로 올림 → 끝음을 자연스럽게 내리도록 명시
    "caster_m40_v2": ("A Korean male baseball play-by-play announcer in his 40s, deep and resonant voice, "
                      "upbeat and energetic, brisk pace. Natural Korean broadcast intonation: "
                      "sentence endings fall naturally, never rising or sing-song.", 3),
    # v1은 소리를 지르는 느낌 → 활기는 유지하되 보통 음량, 차분한 힘
    "caster_m20_v2": ("A Korean male sports announcer in his late 20s, bright and clear voice, "
                      "lively but composed delivery at a normal speaking volume, never shouting, "
                      "friendly professional broadcast tone.", 4),
}

refs, design = {}, None
for name, (desc, seed) in VOICES.items():
    path = OUT / f"ref_{name}.wav"
    if path.exists():
        wav, sr = sf.read(path, dtype="float32")
        print(f"{name}: 저장된 참조 음성 사용")
    else:
        if design is None:
            design = load("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")
        torch.manual_seed(seed)
        with Timer() as t:
            wavs, sr = design.generate_voice_design(text=REF_TEXT, language="Korean", instruct=desc)
        wav = wavs[0]
        sf.write(path, wav, sr)
        print(f"{name}: {len(wav) / sr:.1f}초 음성, 생성 {t.sec:.1f}초")
    refs[name] = (wav, sr)
    display(Audio(wav, rate=sr))

if design is not None:
    del design
    free()

In [ ]:
# 2) Base 모델에 참조 음성을 등록해서 목소리 고정
clone = load("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
prompts = {name: clone.create_voice_clone_prompt(ref_audio=ref, ref_text=REF_TEXT) for name, ref in refs.items()}

# 04 템플릿 미리보기에서 가져온 문장 + 읽기 어려운 것들 (숫자, 용어, 외국인 이름)
TEST = [
    "1회초, KIA 공격입니다. 스코어 KIA 0 대 SSG 0.",
    "마운드에는 화이트.",
    "노아웃, 타석에는 김호령.",
    "자, 바깥쪽 빠졌고요.",
    "151킬로 직구, 헛스윙 나옵니다.",
    "먹혔고 파울입니다.",
    "볼카운트 투앤원.",
    "풀카운트.",
    "스위퍼, 낮은 공 골라냅니다.",
    "자, 볼넷, 1루로 걸어 나갑니다.",
    "원아웃, 타석에는 카스트로.",
    "우중간 빠집니다, 2루타.",
    "투수 피치클락 위반, 볼.",
    "큰 타구! 넘어갑니다! 홈런!",
    "쓰리아웃, 공수 교대합니다.",
]

In [ ]:
# 2-1) TTS 입력 직전에만 읽는 대로 바꾼다 (대본 원문은 그대로 둠)
# KIA는 영어 단어처럼 읽혀서 '기아'로. 숫자·SSG 등은 그대로도 잘 읽음 (1차 청취 확인)
READ_AS = {"KIA": "기아"}


def tts_text(text):
    for src, dst in READ_AS.items():
        text = text.replace(src, dst)
    return text


# 아직 안 들어본 팀명 약어도 확인
TEST += [
    "8회말, KT 공격입니다. 스코어 LG 3 대 KT 2.",
    "스코어 NC 1 대 두산 4.",
]
TEST = [tts_text(t) for t in TEST]
print(TEST[0])

In [ ]:
# 3) 한 문장씩 합성: 발음 확인 + 문장 단위 속도
VOICE = "caster_m30"
GAP = 0.4  # 문장 사이 무음(초)

wavs_single, gen_sec = [], 0.0
for text in TEST:
    with Timer() as t:
        wavs, sr = clone.generate_voice_clone(text=text, language="Korean", voice_clone_prompt=prompts[VOICE])
    wavs_single.append(wavs[0])
    gen_sec += t.sec
    print(f"{len(wavs[0]) / sr:4.1f}초 ({t.sec:4.1f}초 걸림) | {text}")

silence = np.zeros(int(GAP * sr), dtype=wavs_single[0].dtype)
joined = np.concatenate([w for wav in wavs_single for w in (wav, silence)])
audio_sec = len(joined) / sr
sf.write(OUT / f"script_{VOICE}.wav", joined, sr)

print(f"\n음성 {audio_sec:.1f}초 / 합성 {gen_sec:.1f}초 → RTF {gen_sec / audio_sec:.2f} (1보다 작으면 실시간보다 빠름)")
display(Audio(joined, rate=sr))

In [ ]:
# 4) 배치 합성: 같은 문장들을 한 번에 → 속도가 얼마나 빨라지는지
with Timer() as t:
    wavs_batch, sr = clone.generate_voice_clone(
        text=TEST, language=["Korean"] * len(TEST), voice_clone_prompt=prompts[VOICE],
    )
batch_audio = sum(len(w) for w in wavs_batch) / sr + GAP * len(TEST)
rtf_batch = t.sec / batch_audio
print(f"배치 {len(TEST)}문장: 음성 {batch_audio:.1f}초 / 합성 {t.sec:.1f}초 → RTF {rtf_batch:.2f}")

# 배치 결과도 목소리·발음이 같은지 확인
display(Audio(np.concatenate([w for wav in wavs_batch for w in (wav, silence)]), rate=sr))

In [ ]:
# 5) 나머지 목소리도 같은 대본으로 (화자 다양성 확인)
for name in VOICES:
    if name == VOICE:
        continue
    wavs, sr = clone.generate_voice_clone(text=TEST[:8], language=["Korean"] * 8, voice_clone_prompt=prompts[name])
    print(name)
    display(Audio(np.concatenate([w for wav in wavs for w in (wav, silence)]), rate=sr))

In [ ]:
# 6) 데이터셋 전체 합성 시간 추정
# 반 이닝 평균: 이벤트 약 17개 + 투수·타자 소개 약 6개 + 카운트 재언급 약 4개 ≈ 발화 27개
UTTS_PER_CLIP = 27
sec_per_utt = batch_audio / len(TEST)
clip_audio = UTTS_PER_CLIP * sec_per_utt

print(f"클립 1개 ≈ 음성 {clip_audio / 60:.1f}분")
for n in [350, 1000, 3000]:
    hours = n * clip_audio * rtf_batch / 3600
    print(f"클립 {n:>5}개: 음성 {n * clip_audio / 3600:5.1f}시간, 합성 약 {hours:5.1f}시간 (배치 기준)")

## (선택) 0.6B 비교
1.7B가 너무 느리면 아래를 실행. 같은 참조 음성으로 품질과 속도를 비교한다.

In [ ]:
del clone
free()
small = load("Qwen/Qwen3-TTS-12Hz-0.6B-Base")
prompt_small = small.create_voice_clone_prompt(ref_audio=refs[VOICE], ref_text=REF_TEXT)

with Timer() as t:
    wavs_small, sr = small.generate_voice_clone(text=TEST, language=["Korean"] * len(TEST), voice_clone_prompt=prompt_small)
small_audio = sum(len(w) for w in wavs_small) / sr + GAP * len(TEST)
print(f"0.6B 배치: 음성 {small_audio:.1f}초 / 합성 {t.sec:.1f}초 → RTF {t.sec / small_audio:.2f}")
display(Audio(np.concatenate([w for wav in wavs_small for w in (wav, silence)]), rate=sr))